# LeetCode 2013 — Detect Squares bằng Python

Link đề: https://leetcode.com/problems/detect-squares/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Design, Counting

Skeleton LeetCode:

```python
class DetectSquares:

    def __init__(self):
        ...

    def add(self, point: List[int]) -> None:
        ...

    def count(self, point: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Thiết kế một **cấu trúc dữ liệu** hỗ trợ hai thao tác `add` / `count`, và biết cân bằng chi phí giữa chúng.
2. Mô tả hình vuông **song song trục** bằng hai đại lượng: **góc đối diện** (đường chéo) hoặc **cạnh chung một cột** + độ dài cạnh `d`.
3. Đếm số cách chọn **có lặp** (điểm trùng được tính là khác nhau) bằng **tích các tần suất**.
4. Cài bộ chạy chuỗi lệnh kiểu LeetCode và **đối chiếu** với cài đặt brute force trên chuỗi lệnh ngẫu nhiên seed cố định.
5. Tránh các bẫy: diện tích bằng 0 (`d = 0`), chỉ xét một phía, coi điểm trùng là một.

## 2. Đề bài & ví dụ

Có một luồng điểm trên mặt phẳng. Thiết kế lớp `DetectSquares`:

- `add(point)`: thêm điểm `[x, y]` vào cấu trúc. **Cho phép điểm trùng**, và các điểm trùng được coi là **khác nhau**.
- `count(point)`: với điểm truy vấn `[x, y]` (không được thêm vào cấu trúc), đếm số cách chọn **ba điểm** trong cấu trúc sao cho ba điểm đó cùng điểm truy vấn tạo thành một **hình vuông song song trục** có **diện tích dương**.

**Ví dụ:**

```
["DetectSquares","add","add","add","count","count","add","count"]
[[],[[3,10]],[[11,2]],[[3,2]],[[11,10]],[[14,8]],[[11,2]],[[11,10]]]
→ [null,null,null,null,1,0,null,2]
```

- `count([11,10])` = 1: chọn `(3,10), (11,2), (3,2)`.
- `count([14,8])` = 0.
- Sau khi thêm `(11,2)` lần hai, `count([11,10])` = 2 (hai bản sao của `(11,2)` cho hai cách chọn).

**Ràng buộc:** `0 ≤ x, y ≤ 1000`; tổng cộng tối đa `3000` lời gọi `add` và `count`.

In [1]:
import random
import time
from collections import Counter, defaultdict
from itertools import combinations
from typing import List

ops_example = ["DetectSquares", "add", "add", "add", "count", "count", "add", "count"]
args_example = [[], [[3, 10]], [[11, 2]], [[3, 2]], [[11, 10]], [[14, 8]], [[11, 2]], [[11, 10]]]
expected_example = [None, None, None, None, 1, 0, None, 2]

def run_ops(cls, ops, args):
    """Bộ chạy chuỗi lệnh kiểu LeetCode."""
    obj = cls(*args[0])                 # lệnh đầu tiên luôn là hàm khởi tạo
    out = [None]
    for op, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, op)(*a))
    return out
print("Ví dụ:", list(zip(ops_example, args_example)))

Ví dụ: [('DetectSquares', []), ('add', [[3, 10]]), ('add', [[11, 2]]), ('add', [[3, 2]]), ('count', [[11, 10]]), ('count', [[14, 8]]), ('add', [[11, 2]]), ('count', [[11, 10]])]


## 3. Kiến thức nền

### 3.1 Hình vuông song song trục

Bốn đỉnh có dạng `(x1, y1), (x2, y1), (x1, y2), (x2, y2)` với `|x2 − x1| = |y2 − y1| = d > 0`. Nếu biết **một đỉnh** `(x1, y1)` và **đỉnh đối diện** `(x2, y2)` (thỏa `|dx| = |dy| ≠ 0`) thì hai đỉnh còn lại xác định duy nhất: `(x1, y2)` và `(x2, y1)`.

In [2]:
def other_corners(p, q):
    (x1, y1), (x2, y2) = p, q
    if abs(x2 - x1) != abs(y2 - y1) or x1 == x2:
        return None                       # không phải đường chéo của hình vuông dương
    return (x1, y2), (x2, y1)

print(other_corners((11, 10), (3, 2)), other_corners((11, 10), (3, 3)), other_corners((1, 1), (1, 1)))

((11, 2), (3, 10)) None None


### 3.2 Đếm có lặp = tích tần suất

Nếu đỉnh A có `a` bản, đỉnh B có `b` bản, đỉnh C có `c` bản (các điểm trùng tính là khác nhau) thì số cách chọn bộ `(A, B, C)` là `a · b · c`.

In [3]:
cnt = Counter([(3, 10), (11, 2), (3, 2), (11, 2)])
print(cnt, "-> số cách:", cnt[(3, 10)] * cnt[(11, 2)] * cnt[(3, 2)])

Counter({(11, 2): 2, (3, 10): 1, (3, 2): 1}) -> số cách: 2


### 3.3 `defaultdict(Counter)` — bảng hai tầng

`col[x][y]` = số điểm tại `(x, y)`, gom theo cột `x`. Duyệt `col[x]` cho ta mọi điểm **cùng cột** với điểm truy vấn — rất hữu ích để chọn cạnh dọc của hình vuông.

In [4]:
col = defaultdict(Counter)
for x, y in [(3, 10), (11, 2), (3, 2), (11, 2)]:
    col[x][y] += 1
print({x: dict(c) for x, c in col.items()}, "| col[11][2] =", col[11][2], "| col[5][5] =", col[5][5])

{3: {10: 1, 2: 1}, 11: {2: 2}} | col[11][2] = 2 | col[5][5] = 0


### 3.4 Thiết kế lớp và bộ chạy lệnh

LeetCode gọi `obj = DetectSquares()` rồi lần lượt `obj.add(...)`, `obj.count(...)`. Hàm `run_ops` ở trên mô phỏng điều đó, trả về danh sách kết quả (`None` cho hàm không trả gì) để so với `expected`.

In [5]:
class Echo:
    def __init__(self): self.items = []
    def add(self, p): self.items.append(p)
    def count(self, p): return len(self.items)
print(run_ops(Echo, ["Echo", "add", "count", "add", "count"], [[], [[1, 1]], [[0, 0]], [[2, 2]], [[0, 0]]]))

[None, None, 1, None, 2]


## 4. Ý tưởng nền tảng / trực giác

Với truy vấn `q = (11, 10)` và các điểm `(3,10), (11,2), (3,2)`:

- Chọn **đỉnh đối diện** `(3, 2)`: `dx = −8`, `dy = −8` ⇒ `|dx| = |dy| = 8` ✓. Hai đỉnh còn lại: `(11, 2)` và `(3, 10)` — đều có ⇒ 1 hình vuông.
- Không chọn được đỉnh đối diện nào khác.

Sau khi thêm `(11, 2)` lần nữa: đỉnh `(11, 2)` có **2 bản** ⇒ `1 · 2 · 1 = 2` cách.

Mỗi hình vuông chứa `q` có **đúng một** đỉnh đối diện `q`, nên "đếm theo đỉnh đối diện" không đếm trùng.

In [6]:
q = (11, 10)
pts = Counter([(3, 10), (11, 2), (3, 2), (11, 2)])
for p in pts:
    oc = other_corners(q, p)
    if oc:
        a, b = oc
        print(f"đối diện {p} (x{pts[p]}): cần {a} (x{pts[a]}) và {b} (x{pts[b]}) -> {pts[p] * pts[a] * pts[b]} cách")

đối diện (3, 2) (x1): cần (11, 2) (x2) và (3, 10) (x1) -> 2 cách


## 5. Cách 1 — Brute Force / baseline: thử mọi bộ ba điểm

**Ý tưởng:** lưu danh sách điểm (kể cả trùng). `count` duyệt mọi bộ ba chỉ số `i < j < k` và kiểm tra bốn điểm (ba điểm + truy vấn) có tạo hình vuông dương song song trục không.

Kiểm tra 4 điểm là hình vuông song song trục: tập các `x` có đúng 2 giá trị, tập `y` có đúng 2 giá trị, hai khoảng cách bằng nhau và > 0, và **4 điểm là đúng 4 góc** (đa tập điểm bằng đa tập 4 góc).

In [7]:
def is_axis_square(pts4):
    xs = sorted({p[0] for p in pts4}); ys = sorted({p[1] for p in pts4})
    if len(xs) != 2 or len(ys) != 2 or xs[1] - xs[0] != ys[1] - ys[0]:
        return False
    corners = Counter((x, y) for x in xs for y in ys)
    return Counter(map(tuple, pts4)) == corners

class DetectSquaresBrute:
    def __init__(self):
        self.pts = []
    def add(self, point: List[int]) -> None:
        self.pts.append(tuple(point))
    def count(self, point: List[int]) -> int:
        q = tuple(point)
        return sum(1 for a, b, c in combinations(self.pts, 3) if is_axis_square([q, a, b, c]))

print(run_ops(DetectSquaresBrute, ["DetectSquaresBrute"] + ops_example[1:], args_example), "| kỳ vọng", expected_example)

[None, None, None, None, 1, 0, None, 2] | kỳ vọng [None, None, None, None, 1, 0, None, 2]


**Độ phức tạp:** `add` `O(1)`; `count` `O(m³)` với `m` = số điểm đã thêm. Với `m = 3000`: `4,5·10⁹` bộ ba ⇒ không thể.

**Nút thắt:** ba điểm không độc lập — chọn **một** điểm (đỉnh đối diện) là xác định hai điểm còn lại. Ta chỉ cần **đếm** chứ không cần duyệt từng bản sao.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đếm theo đường chéo):** với truy vấn `q = (x, y)`, duyệt mỗi **điểm khác nhau** `p = (px, py)` đang có. **Nếu** `|px − x| = |py − y| ≠ 0` **thì** cộng `cnt[p] · cnt[(x, py)] · cnt[(px, y)]`. Chi phí `O(số điểm khác nhau)`.

**Insight 2 (đếm theo cạnh dọc cùng cột):** chỉ duyệt các điểm **cùng cột** `x` với `q`: `(x, y2)`, `d = y2 − y`, `d ≠ 0`. Đặt `side = |d|`; hình vuông có thể nằm **bên phải** (cột `x + side`) hoặc **bên trái** (cột `x − side`). Với mỗi phía, cộng `col[x][y2] · col[x'][y] · col[x'][y2]`. Chi phí `O(số điểm trong cột x)`.

**Invariant:** `cnt` (hoặc `col`) luôn phản ánh đúng đa tập các điểm đã `add`. `count` **không** thay đổi cấu trúc.

**Quy tắc quan trọng:** `d = 0` ⇒ bỏ qua (diện tích 0); xét **cả hai** phía.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` điểm, duyệt đỉnh đối diện

**Giải thích từng đoạn:**
1. `self.cnt = Counter()` đếm theo tuple `(x, y)`.
2. `add`: `cnt[(x, y)] += 1`.
3. `count(q)`: duyệt `(px, py), c in cnt.items()`; nếu là đường chéo hợp lệ thì cộng `c · cnt[(x, py)] · cnt[(px, y)]`.

Lưu ý: `cnt[...]` với khóa chưa có trả `0` **nhưng** với `Counter` thì không thêm khóa — an toàn khi đang duyệt `items()`. Để chắc chắn, dùng `cnt.get(key, 0)`.

In [8]:
class DetectSquaresDiag:
    def __init__(self):
        self.cnt = Counter()

    def add(self, point: List[int]) -> None:
        self.cnt[tuple(point)] += 1

    def count(self, point: List[int]) -> int:
        x, y = point
        get = self.cnt.get
        ans = 0
        for (px, py), c in self.cnt.items():
            if px != x and abs(px - x) == abs(py - y):
                ans += c * get((x, py), 0) * get((px, y), 0)
        return ans

print(run_ops(DetectSquaresDiag, ["DetectSquaresDiag"] + ops_example[1:], args_example))

[None, None, None, None, 1, 0, None, 2]


**Độ phức tạp Cách 2:** `add` `O(1)`; `count` `O(K)` với `K` = số điểm **khác nhau** (≤ 3000). Tổng worst case `3000 · 3000 ≈ 9·10⁶` — chạy tốt.

### 7.2 Cách 3 — Gom theo cột, duyệt cạnh dọc (bản nộp chính)

**Giải thích từng đoạn:**
1. `self.col = defaultdict(Counter)`: `col[x][y]`.
2. `count(q = (x, y))`: nếu cột `x` rỗng ⇒ `0`.
3. Với mỗi `y2, c in col[x].items()` mà `y2 ≠ y`: `side = abs(y2 − y)`; với `x2 ∈ {x + side, x − side}`: cộng `c · col[x2][y] · col[x2][y2]` (dùng `.get` để không tạo cột rỗng).

In [9]:
class DetectSquaresColumn:
    def __init__(self):
        self.col = defaultdict(Counter)

    def add(self, point: List[int]) -> None:
        x, y = point
        self.col[x][y] += 1

    def count(self, point: List[int]) -> int:
        x, y = point
        if x not in self.col:
            return 0
        ans = 0
        for y2, c in self.col[x].items():
            if y2 == y:
                continue
            side = abs(y2 - y)
            for x2 in (x + side, x - side):
                other = self.col.get(x2)
                if other:
                    ans += c * other.get(y, 0) * other.get(y2, 0)
        return ans

print(run_ops(DetectSquaresColumn, ["DetectSquaresColumn"] + ops_example[1:], args_example))

[None, None, None, None, 1, 0, None, 2]


**Độ phức tạp Cách 3:** `add` `O(1)`; `count` `O(số điểm khác nhau trong cột x)` — worst case vẫn `O(K)` (mọi điểm cùng cột), nhưng thường nhỏ hơn nhiều. Space `O(K)`.

### 7.3 Cách 4 — Lưới đếm 1001×1001, duyệt độ dài cạnh

Vì tọa độ ≤ 1000, dùng mảng hai chiều `grid[x][y]`. `count` duyệt mọi `d = 1..1000` và 4 hướng (dấu của `dx`, `dy`). Chi phí cố định `O(4·1000)` mỗi truy vấn, không phụ thuộc số điểm — hợp khi có rất nhiều điểm.

In [10]:
class DetectSquaresGrid:
    N = 1001
    def __init__(self):
        self.g = [[0] * self.N for _ in range(self.N)]

    def add(self, point: List[int]) -> None:
        x, y = point
        self.g[x][y] += 1

    def count(self, point: List[int]) -> int:
        x, y = point
        g, N, ans = self.g, self.N, 0
        for sx in (1, -1):
            for sy in (1, -1):
                for d in range(1, N):
                    x2, y2 = x + sx * d, y + sy * d
                    if not (0 <= x2 < N and 0 <= y2 < N):
                        break
                    if g[x2][y2]:
                        ans += g[x2][y2] * g[x][y2] * g[x2][y]
        return ans

print(run_ops(DetectSquaresGrid, ["DetectSquaresGrid"] + ops_example[1:], args_example))

[None, None, None, None, 1, 0, None, 2]


**Độ phức tạp Cách 4:** `add` `O(1)`; `count` `O(4·1000)`; Space `O(1001²) ≈ 10⁶` ô — tốn bộ nhớ hơn nhiều.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ, ở lần `count([11,10])` cuối (sau 4 lần `add`). `col = {3: {10:1, 2:1}, 11: {2:2}}`.

| y2 trong cột 11 | c | side | x2 | col[x2][10] | col[x2][y2] | cộng |
|---|---|---|---|---|---|---|
| 2 | 2 | 8 | 19 | (không có cột) | — | 0 |
| 2 | 2 | 8 | 3 | 1 | col[3][2] = 1 | 2·1·1 = 2 |

Tổng `2` ✓.

In [11]:
class DetectSquaresTrace(DetectSquaresColumn):
    def count(self, point):
        x, y = point
        print(f"count({point}): cột {x} = {dict(self.col.get(x, {}))}")
        ans = 0
        for y2, c in self.col.get(x, {}).items():
            if y2 == y:
                print(f"  y2={y2}: trùng hàng với truy vấn -> bỏ (d = 0)"); continue
            side = abs(y2 - y)
            for x2 in (x + side, x - side):
                other = self.col.get(x2, {})
                add = c * other.get(y, 0) * other.get(y2, 0)
                ans += add
                print(f"  y2={y2} (x{c}), side={side}, x2={x2}: col[x2][{y}]={other.get(y, 0)}, "
                      f"col[x2][{y2}]={other.get(y2, 0)} -> +{add}, ans={ans}")
        return ans

print(run_ops(DetectSquaresTrace, ["DetectSquaresTrace"] + ops_example[1:], args_example))

count([11, 10]): cột 11 = {2: 1}
  y2=2 (x1), side=8, x2=19: col[x2][10]=0, col[x2][2]=0 -> +0, ans=0
  y2=2 (x1), side=8, x2=3: col[x2][10]=1, col[x2][2]=1 -> +1, ans=1
count([14, 8]): cột 14 = {}
count([11, 10]): cột 11 = {2: 2}
  y2=2 (x2), side=8, x2=19: col[x2][10]=0, col[x2][2]=0 -> +0, ans=0
  y2=2 (x2), side=8, x2=3: col[x2][10]=1, col[x2][2]=1 -> +2, ans=2
[None, None, None, None, 1, 0, None, 2]


## 9. Code nộp LeetCode

In [12]:
class DetectSquares:

    def __init__(self):
        self.col = defaultdict(Counter)          # col[x][y] = số điểm tại (x, y)

    def add(self, point: List[int]) -> None:
        x, y = point
        self.col[x][y] += 1

    def count(self, point: List[int]) -> int:
        x, y = point
        column = self.col.get(x)
        if not column:
            return 0
        ans = 0
        for y2, c in column.items():             # cạnh dọc (x, y) - (x, y2)
            if y2 == y:
                continue                         # diện tích 0
            side = abs(y2 - y)
            for x2 in (x + side, x - side):      # hình vuông bên phải / bên trái
                other = self.col.get(x2)
                if other:
                    ans += c * other.get(y, 0) * other.get(y2, 0)
        return ans

assert run_ops(DetectSquares, ops_example, args_example) == expected_example
print("Ví dụ đề: OK", run_ops(DetectSquares, ops_example, args_example))

Ví dụ đề: OK [None, None, None, None, 1, 0, None, 2]


## 10. Vì sao thuật toán đúng

**Song ánh.** Mỗi bộ ba điểm (theo chỉ số, kể cả bản trùng) cùng `q = (x, y)` tạo hình vuông dương song song trục ⇔ có `side > 0` và một phía `x2 = x ± side` sao cho ba điểm lần lượt là `(x, y2)`, `(x2, y)`, `(x2, y2)` với `|y2 − y| = side`.

- Đỉnh cùng cột với `q` là **duy nhất** trong hình vuông (`(x, y2)`), nên mỗi hình vuông được sinh ra **đúng một lần** bởi cặp `(y2, x2)` tương ứng. Hai phía `x + side` và `x − side` khác nhau vì `side > 0`.
- Với một cặp `(y2, x2)` cố định, số bộ ba = `col[x][y2] · col[x2][y] · col[x2][y2]` (chọn độc lập một bản sao cho mỗi đỉnh).

**Không đếm thừa:** `y2 = y` bị loại ⇒ không có hình vuông diện tích 0; `q` không được thêm vào cấu trúc nên không tự ghép với chính nó. Cấu trúc `col` đúng là đa tập điểm đã thêm ⇒ đáp án đúng.

## 11. Phân tích Time / Space

Gọi `m` = số lần `add`, `K` = số điểm khác nhau (`K ≤ m ≤ 3000`), `c_x` = số điểm khác nhau trong cột `x`.

| Cách | `add` | `count` | Space |
|---|---|---|---|
| 1. Brute force | `O(1)` | `O(m³)` | `O(m)` |
| 2. Duyệt đường chéo | `O(1)` | `O(K)` | `O(K)` |
| 3. Duyệt cột | `O(1)` | `O(c_x)` ≤ `O(K)` | `O(K)` |
| 4. Lưới | `O(1)` | `O(4000)` | `O(10⁶)` |

Đo trên 3000 lệnh ngẫu nhiên (1500 `add` + 1500 `count`), tọa độ nhỏ để có nhiều hình vuông:

In [13]:
rng = random.Random(2013)
ops, args = ["DetectSquares"], [[]]
for _ in range(3000):
    p = [rng.randint(0, 30), rng.randint(0, 30)]
    ops.append(rng.choice(["add", "count"])); args.append([p])
for cls in (DetectSquaresDiag, DetectSquaresColumn, DetectSquaresGrid, DetectSquares):
    t = time.perf_counter(); out = run_ops(cls, [cls.__name__] + ops[1:], args)
    print(f"{cls.__name__:20s}: tổng count = {sum(v for v in out if v)}  ({time.perf_counter() - t:.3f}s)")

DetectSquaresDiag   : tổng count = 54206  (0.053s)
DetectSquaresColumn : tổng count = 54206  (0.014s)


DetectSquaresGrid   : tổng count = 54206  (0.185s)
DetectSquares       : tổng count = 54206  (0.006s)


## 12. Bộ test

- Ví dụ đề.
- Biên: truy vấn trên cấu trúc rỗng; truy vấn trùng với điểm đã có (không được tự tạo hình vuông); nhiều bản trùng; hình vuông ở cả 4 hướng; hình chữ nhật (không phải vuông) không được đếm.
- Đối chiếu mọi cài đặt với brute force trên chuỗi lệnh ngẫu nhiên seed cố định.

In [14]:
classes = [DetectSquaresBrute, DetectSquaresDiag, DetectSquaresColumn, DetectSquaresGrid, DetectSquares]
def run_all(ops, args):
    outs = [run_ops(cls, [cls.__name__] + ops[1:], args) for cls in classes]
    for o in outs[1:]:
        assert o == outs[0], (outs[0], o)
    return outs[0]

assert run_all(ops_example, args_example) == expected_example
# rỗng
assert run_all(["D", "count"], [[], [[5, 5]]]) == [None, 0]
# truy vấn trùng điểm đã có: không tự ghép
assert run_all(["D", "add", "add", "count"], [[], [[1, 1]], [[1, 1]], [[1, 1]]]) == [None, None, None, 0]
# 4 hướng quanh (5,5) với cạnh 2, mỗi góc 1 bản
pts = [[x, y] for x in (3, 5, 7) for y in (3, 5, 7) if (x, y) != (5, 5)]
assert run_all(["D"] + ["add"] * len(pts) + ["count"], [[]] + [[p] for p in pts] + [[[5, 5]]])[-1] == 4
# hình chữ nhật 2x3: không đếm
assert run_all(["D", "add", "add", "add", "count"], [[], [[0, 3]], [[2, 0]], [[2, 3]], [[0, 0]]])[-1] == 0
# nhiều bản trùng: 2*3*4
seq = [[0, 1]] * 2 + [[1, 0]] * 3 + [[1, 1]] * 4
assert run_all(["D"] + ["add"] * 9 + ["count"], [[]] + [[p] for p in seq] + [[[0, 0]]])[-1] == 24
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [15]:
rng = random.Random(42)
for trial in range(120):
    ops, args = ["D"], [[]]
    R = rng.choice([2, 3, 5])
    for _ in range(rng.randint(1, 30)):
        ops.append("add" if rng.random() < 0.6 else "count")
        args.append([[rng.randint(0, R), rng.randint(0, R)]])
    run_all(ops, args)
print("120 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force")

120 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `set`, coi điểm trùng là một

Đề nói rõ điểm trùng là **khác nhau**. Ví dụ đề: lần `count` cuối phải ra `2`, bản dùng `set` ra `1`.

In [16]:
class WrongSet:
    def __init__(self): self.s = set()
    def add(self, p): self.s.add(tuple(p))
    def count(self, p):
        x, y = p; ans = 0
        for px, py in self.s:
            if px != x and abs(px - x) == abs(py - y):
                ans += ((x, py) in self.s) * ((px, y) in self.s)   # SAI: bỏ qua số lần lặp
        return ans

print("sai:", run_ops(WrongSet, ["WrongSet"] + ops_example[1:], args_example)[-1], "| đúng:", expected_example[-1])

sai: 1 | đúng: 2


In [17]:
print("sửa (đếm tần suất):", run_ops(DetectSquares, ops_example, args_example)[-1])

sửa (đếm tần suất): 2


### Lỗi 2 — Không loại `d = 0` (diện tích bằng 0)

Nếu cho `y2 == y`, thì `side = 0`, `x2 = x` và ta nhân `col[x][y]³` — đếm các "hình vuông" suy biến khi truy vấn trùng điểm đã có.

In [18]:
class WrongZero(DetectSquaresColumn):
    def count(self, point):
        x, y = point; ans = 0
        for y2, c in self.col.get(x, {}).items():          # SAI: không bỏ y2 == y
            side = abs(y2 - y)
            for x2 in {x + side, x - side}:
                o = self.col.get(x2, {})
                ans += c * o.get(y, 0) * o.get(y2, 0)
        return ans

ops2, args2 = ["D", "add", "add", "count"], [[], [[1, 1]], [[1, 1]], [[1, 1]]]
print("sai:", run_ops(WrongZero, ops2, args2)[-1], "| đúng:", run_ops(DetectSquares, ops2, args2)[-1])

sai: 8 | đúng: 0


In [19]:
print("sửa (bỏ y2 == y):", run_ops(DetectSquares, ops2, args2)[-1])

sửa (bỏ y2 == y): 0


### Lỗi 3 — Chỉ xét hình vuông ở một phía

Chỉ thử `x2 = x + side` sẽ bỏ sót hình vuông nằm bên trái điểm truy vấn — chính là trường hợp của ví dụ đề (`x2 = 3 < 11`).

In [20]:
class WrongOneSide(DetectSquaresColumn):
    def count(self, point):
        x, y = point; ans = 0
        for y2, c in self.col.get(x, {}).items():
            if y2 == y: continue
            x2 = x + abs(y2 - y)                               # SAI: thiếu x - side
            o = self.col.get(x2, {})
            ans += c * o.get(y, 0) * o.get(y2, 0)
        return ans

print("sai:", run_ops(WrongOneSide, ["W"] + ops_example[1:], args_example), "| đúng:", expected_example)

sai: [None, None, None, None, 0, 0, None, 0] | đúng: [None, None, None, None, 1, 0, None, 2]


In [21]:
print("sửa:", run_ops(DetectSquares, ops_example, args_example))

sửa: [None, None, None, None, 1, 0, None, 2]


### Lỗi 4 — Quên kiểm `|dx| == |dy|` khi duyệt đường chéo

Duyệt đỉnh đối diện mà chỉ kiểm `px != x and py != y` sẽ đếm cả **hình chữ nhật**.

In [22]:
class WrongRect(DetectSquaresDiag):
    def count(self, point):
        x, y = point; get = self.cnt.get; ans = 0
        for (px, py), c in self.cnt.items():
            if px != x and py != y:                            # SAI: thiếu |dx| == |dy|
                ans += c * get((x, py), 0) * get((px, y), 0)
        return ans

ops3, args3 = ["D", "add", "add", "add", "count"], [[], [[0, 3]], [[2, 0]], [[2, 3]], [[0, 0]]]
print("sai:", run_ops(WrongRect, ops3, args3)[-1], "| đúng:", run_ops(DetectSquares, ops3, args3)[-1])

sai: 1 | đúng: 0


In [23]:
print("sửa:", run_ops(DetectSquaresDiag, ops3, args3)[-1])

sửa: 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài thiết kế có `add` + `count/query`** ⇒ hỏi: thao tác nào nhiều hơn, chi phí nào chấp nhận được? Ở đây `add` `O(1)`, `count` `O(K)` là đủ.
- **Hình vuông/chữ nhật song song trục** ⇒ chọn **một** đỉnh hoặc **một cạnh** là xác định phần còn lại ⇒ tra bảng băm.
- "Điểm trùng tính là khác nhau" ⇒ đếm bằng **tích tần suất**.
- Với `.get(key, 0)` thay vì `[key]` trên `defaultdict` để **không tạo khóa rác** khi đang duyệt.
- Luôn kiểm: diện tích 0, cả hai/bốn hướng, hình chữ nhật không phải vuông.

## 15. Biến thể của bài

1. **Đếm hình chữ nhật song song trục** có `q` là một đỉnh (bài tập 5): bỏ điều kiện `|dx| = |dy|`.
2. **LeetCode 939 — Minimum Area Rectangle:** từ tập điểm, tìm hình chữ nhật diện tích nhỏ nhất — duyệt cặp đường chéo + `set`.
3. **LeetCode 963 — Minimum Area Rectangle II:** hình chữ nhật **xoay tùy ý** — gom các cặp điểm theo (trung điểm, độ dài đường chéo).
4. **Hỗ trợ `remove(point)`:** chỉ cần giảm tần suất; công thức `count` giữ nguyên.
5. **Đếm mọi hình vuông** trong tập điểm (không có truy vấn): cộng `count(p)` cho mọi điểm rồi chia 4 (cẩn thận với điểm trùng).

In [24]:
class DetectSquaresRemovable(DetectSquares):
    def remove(self, point):
        x, y = point
        self.col[x][y] -= 1
        if self.col[x][y] == 0:
            del self.col[x][y]
            if not self.col[x]:
                del self.col[x]

class BruteRemovable(DetectSquaresBrute):
    def remove(self, point):
        self.pts.remove(tuple(point))

rng = random.Random(4)
for _ in range(80):
    a, b = DetectSquaresRemovable(), BruteRemovable(); added = []
    for _ in range(rng.randint(1, 25)):
        r = rng.random(); p = [rng.randint(0, 3), rng.randint(0, 3)]
        if r < 0.5: a.add(p); b.add(p); added.append(p)
        elif r < 0.7 and added:
            p = added.pop(rng.randrange(len(added))); a.remove(p); b.remove(p)
        else: assert a.count(p) == b.count(p)
print("Biến thể có remove (tự dựng) khớp brute force")

Biến thể có remove (tự dựng) khớp brute force


## 16. So sánh các cách

| Cách | Ý tưởng | `count` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mọi bộ ba | `O(m³)` | `O(m)` | kiểm tra |
| 2. Đỉnh đối diện | duyệt mọi điểm khác nhau | `O(K)` | `O(K)` | dễ hiểu, tổng quát |
| 3. Cạnh cùng cột | duyệt điểm trong cột `x` | `O(c_x)` | `O(K)` | **nộp bài**, nhanh trong thực tế |
| 4. Lưới | duyệt `d` và 4 hướng | `O(4·1000)` | `O(10⁶)` | rất nhiều điểm, tọa độ nhỏ |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** thêm `(0,0), (0,1), (1,0), (1,1), (1,1)` rồi `count([0,0])`; sau đó `count([2,2])`.
2. **Tính tay:** thêm `(1,1), (1,3), (3,1), (5,1), (5,3)` rồi tính `count([3,3])` (có hình vuông ở cả hai phía).
3. **Điền TODO:** `is_square_corners(p, q, r, s)` kiểm 4 điểm là 4 góc hình vuông song song trục, diện tích dương.
4. **Tự viết lại:** viết lại `DetectSquares` (Cách 3) không nhìn code.
5. **Mini challenge:** `DetectRectangles` — `count(q)` đếm số bộ ba tạo **hình chữ nhật** song song trục, diện tích dương, với `q`. Kiểm bằng brute force.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_square_corners(p, q, r, s):
    # TODO
    pass

In [26]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
class MyDetectSquares:
    def __init__(self):
        pass  # TODO
    def add(self, point):
        pass  # TODO
    def count(self, point):
        pass  # TODO

In [27]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
class DetectRectangles:
    def __init__(self):
        pass  # TODO
    def add(self, point):
        pass  # TODO
    def count(self, point):
        pass  # TODO

## 18. Gợi ý 3 mức

**Bài 1.** (1) Với `q = (0,0)`, chỉ có hình vuông cạnh 1. (2) Đỉnh `(1,1)` có 2 bản. (3) `1·1·2 = 2`; `q = (2,2)` không có điểm cùng cột ⇒ `0`.

**Bài 2.** (1) Cột `x = 3` chỉ có `(3,1)`, `side = 2`. (2) Hai phía: `x2 = 5` và `x2 = 1`. (3) Bên phải cần `(5,3)`, `(5,1)`; bên trái cần `(1,3)`, `(1,1)` ⇒ `1 + 1 = 2`.

**Bài 3.** (1) Hai giá trị `x`, hai giá trị `y` khác nhau. (2) Hai khoảng bằng nhau. (3) Đa tập 4 điểm đúng bằng 4 góc.

**Bài 4.** (1) `defaultdict(Counter)`. (2) Bỏ `y2 == y`. (3) Hai phía `x ± side`.

**Bài 5.** (1) Không còn ràng buộc `side`: duyệt `y2` trong cột `x` và **mọi** cột `x2 ≠ x`. (2) Cộng `col[x][y2]·col[x2][y]·col[x2][y2]`. (3) Chi phí `O(c_x · số cột)`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1, 2
d = DetectSquares()
for p in [(0, 0), (0, 1), (1, 0), (1, 1), (1, 1)]: d.add(list(p))
assert d.count([0, 0]) == 2 and d.count([2, 2]) == 0
d2 = DetectSquares()
for p in [(1, 1), (1, 3), (3, 1), (5, 1), (5, 3)]: d2.add(list(p))
assert d2.count([3, 3]) == 2
print("Bài 1: 2 và 0 | Bài 2: 2")

Bài 1: 2 và 0 | Bài 2: 2


In [29]:
# Bài 3, 4
def is_square_corners_ans(p, q, r, s):
    return is_axis_square([p, q, r, s])

assert is_square_corners_ans((0, 0), (0, 2), (2, 0), (2, 2))
assert not is_square_corners_ans((0, 0), (0, 3), (2, 0), (2, 3))
assert not is_square_corners_ans((1, 1), (1, 1), (1, 1), (1, 1))
assert not is_square_corners_ans((0, 0), (0, 2), (2, 0), (0, 0))

MyDetectSquaresAns = DetectSquares
print("Bài 3: OK | Bài 4 (dùng lại lớp nộp):", run_ops(MyDetectSquaresAns, ops_example, args_example))

Bài 3: OK | Bài 4 (dùng lại lớp nộp): [None, None, None, None, 1, 0, None, 2]


In [30]:
# Bài 5
class DetectRectanglesAns:
    def __init__(self):
        self.col = defaultdict(Counter)
    def add(self, point):
        self.col[point[0]][point[1]] += 1
    def count(self, point):
        x, y = point
        column = self.col.get(x)
        if not column: return 0
        ans = 0
        for y2, c in column.items():
            if y2 == y: continue
            for x2, other in self.col.items():
                if x2 != x:
                    ans += c * other.get(y, 0) * other.get(y2, 0)
        return ans

def is_axis_rect(pts4):
    xs = sorted({p[0] for p in pts4}); ys = sorted({p[1] for p in pts4})
    if len(xs) != 2 or len(ys) != 2: return False
    return Counter(map(tuple, pts4)) == Counter((x, y) for x in xs for y in ys)

class RectBrute(DetectSquaresBrute):
    def count(self, point):
        q = tuple(point)
        return sum(1 for a, b, c in combinations(self.pts, 3) if is_axis_rect([q, a, b, c]))

rng = random.Random(55)
for _ in range(100):
    a, b = DetectRectanglesAns(), RectBrute()
    for _ in range(rng.randint(1, 22)):
        p = [rng.randint(0, 3), rng.randint(0, 3)]
        if rng.random() < 0.6: a.add(p); b.add(p)
        else: assert a.count(p) == b.count(p)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
col = defaultdict(Counter)                 # add: col[x][y] += 1
def count(x, y):
    ans = 0
    for y2, c in col.get(x, {}).items():   # cạnh dọc chung cột với truy vấn
        if y2 == y: continue
        s = abs(y2 - y)
        for x2 in (x + s, x - s):          # hai phía
            o = col.get(x2)
            if o: ans += c * o.get(y, 0) * o.get(y2, 0)
    return ans
```

In [31]:
def template_count(col, x, y):
    ans = 0
    for y2, c in col.get(x, {}).items():
        if y2 == y: continue
        s = abs(y2 - y)
        for x2 in (x + s, x - s):
            o = col.get(x2)
            if o: ans += c * o.get(y, 0) * o.get(y2, 0)
    return ans
col = defaultdict(Counter)
for p in [(3, 10), (11, 2), (3, 2), (11, 2)]: col[p[0]][p[1]] += 1
print(template_count(col, 11, 10))

2


**Checklist:**

- [ ] Đếm **tần suất** (không dùng `set`).
- [ ] Bỏ `y2 == y` (diện tích 0).
- [ ] Xét **cả hai** phía `x + side` và `x − side`.
- [ ] Nhân ba tần suất (`c · col[x2][y] · col[x2][y2]`).
- [ ] Dùng `.get` để không tạo khóa rỗng khi đang duyệt.
- [ ] Test bằng bộ chạy lệnh + brute force trên chuỗi lệnh ngẫu nhiên.